# GKG Scalability Test — Exp 1 (BRD + GZR)

**Goal**: Test how large instances the BRD + GZR pipeline can handle.

- **n** (players): [2, 3, 5, 8, 10, 15, 20, 25, 30, 35, 40, 45, 50]
- **m** (items):   [30, 50]
- **cap_factor**:  [0.2, 0.5]
- **corr**:        [uncorrelated, weakly_correlated, strongly_correlated]
- **reps**: 1 (single replication for efficiency)

Total: 13 × 2 × 2 × 3 × 1 = **156 instances**

Only Exp 1 (BRD warm-starts GZR @ alpha=1 with `stop_at_first=False`).

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd

# Ensure we can import the `gkg` package
    



In [ ]:
import json as _json
from gipg.gkg.instance import generate_gkg_instance
from gipg.gkg.heuristics import brd_random_restart
from gipg.gkg.gzr import solve_gzr
from gipg.gkg.social_optimum import solve_social_optimum, compute_pos

RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Separate folder for scalability instances (not mixed with main 10-rep instances)
INST_DIR = Path('../data/gkg') / 'gkg_scalability_instances'
INST_DIR.mkdir(parents=True, exist_ok=True)

print('GKG modules loaded successfully.')
print(f'Instances will be saved to: {INST_DIR}')
print(f'Results  will be saved to:  {RESULTS_DIR}')

## 1. Generate Instances On-the-Fly

In [ ]:
# --- Scalability grid ---
N_VALUES = [2, 3, 5, 8, 10, 15, 20, 25, 30, 35, 40, 45, 50]
M_VALUES = [30, 50]
CAP_FACTORS = [0.2, 0.5]
CORRS = ['uncorrelated', 'weakly_correlated', 'strongly_correlated']
REPS = 1
SEED0 = 54321   # different base seed from the main experiment
R = 1000

instances = []
k = 0
for n in N_VALUES:
    for m in M_VALUES:
        for cf in CAP_FACTORS:
            for corr in CORRS:
                for r in range(REPS):
                    seed = SEED0 + k * 1009
                    inst = generate_gkg_instance(
                        n=n, m=m, cap_factor=cf, corr=corr, seed=seed, R=R,
                    )
                    # Save instance to separate scalability folder
                    fname = f'gkg_n{n}_m{m}_cf{cf}_corr{corr}_rep{r}.json'
                    with open(INST_DIR / fname, 'w', encoding='utf-8') as f:
                        _json.dump(inst.to_dict(), f)
                    instances.append(inst)
                    k += 1

print(f'Generated and saved {len(instances)} instances to {INST_DIR}')

# Summary
from collections import Counter
size_counts = Counter()
for inst in instances:
    size_counts[(inst.n, inst.m)] += 1
print('\n--- By (n, m) ---')
for key in sorted(size_counts.keys()):
    print(f'  n={key[0]}, m={key[1]}: {size_counts[key]} instances')

## 2. Experiment 1: Best PNE and POS (BRD + GZR @ alpha=1)

For each instance:
1. **BRD** warm-starts **GZR @ alpha=1** (if BRD found a PNE)
2. **GZR** with `stop_at_first=False` — optimises to MIP gap=0, finding the **best** PNE (max welfare)
3. **Social Optimum** computed separately
4. **POS** = SO / best_PNE

In [ ]:
TOTAL_TIME_LIMIT = 600.0  # Total budget for BRD + GZR combined
SO_TIME_LIMIT = 600.0     # 10 minutes for social optimum

exp1_path = RESULTS_DIR / 'gkg_scalability_exp1.csv'

results_exp1 = []

for idx, inst in enumerate(instances):
    tag = f'n{inst.n}_m{inst.m}_cf{inst.cap_factor}_corr{inst.corr}'

    row = {
        'tag': tag,
        'n': inst.n,
        'm': inst.m,
        'cap_factor': inst.cap_factor,
        'corr': inst.corr,
        'seed': inst.seed,
    }

    # --- Phase 1: BRD ---
    try:
        x_brd, brd_pne, brd_time = brd_random_restart(
            inst, max_init=3, max_round=15, seed=0,
        )
        row['brd_found_pne'] = brd_pne
        row['brd_time'] = brd_time
        if brd_pne:
            row['initial_pne_welfare'] = float(sum(
                int(inst.p[i, j]) * int(x_brd[i, j])
                for i in range(inst.n) for j in range(inst.m)
            ))
    except Exception as e:
        row['brd_found_pne'] = False
        row['brd_time'] = 0.0
        x_brd = None
        brd_pne = False
        brd_time = 0.0
        print(f'  !! BRD ERROR ({tag}): {e}')

    # --- Phase 2: GZR @ alpha=1, warm-started with BRD profile ---
    gzr_time_limit = max(TOTAL_TIME_LIMIT - brd_time, 60.0)

    warm = x_brd if brd_pne else None
    try:
        gzr_res = solve_gzr(
            inst, alpha=1.0, time_limit=gzr_time_limit,
            warm_start=warm,
            stop_at_first=False, verbose=False,
        )
        row['gzr_status'] = gzr_res.status
        row['gzr_mip_gap'] = gzr_res.mip_gap
        row['gzr_obj_bound'] = gzr_res.obj_bound
        row['gzr_cuts'] = gzr_res.cuts_added
        row['gzr_br_calls'] = gzr_res.br_calls
        row['gzr_time'] = brd_time + gzr_res.runtime   # BRD + GZR combined
        row['gzr_obj_val'] = gzr_res.obj_val
        row['gzr_first_pne_time'] = (brd_time + gzr_res.first_pne_time
                                            if gzr_res.first_pne_time is not None else None)
    except Exception as e:
        gzr_res = None
        row['gzr_status'] = 'ERROR'
        row['gzr_mip_gap'] = None
        row['gzr_obj_bound'] = None
        row['gzr_cuts'] = None
        row['gzr_br_calls'] = None
        row['gzr_time'] = None
        row['gzr_obj_val'] = None
        row['gzr_first_pne_time'] = None
        print(f'  !! GZR ERROR ({tag}): {e}')

    # Best PNE welfare
    if gzr_res is not None and gzr_res.profile is not None:
        row['best_pne_welfare'] = float(sum(
            int(inst.p[i, j]) * int(gzr_res.profile[i, j])
            for i in range(inst.n) for j in range(inst.m)
        ))
    elif brd_pne:
        row['best_pne_welfare'] = row.get('initial_pne_welfare')
    else:
        row['best_pne_welfare'] = None

    # --- Phase 3: Social Optimum ---
    try:
        so_res = solve_social_optimum(inst, time_limit=SO_TIME_LIMIT, verbose=False)
        row['so_status'] = so_res.status
        row['so_welfare'] = so_res.opt_cost
        row['so_time'] = so_res.runtime
    except Exception as e:
        row['so_status'] = 'ERROR'
        row['so_welfare'] = None
        row['so_time'] = None
        print(f'  !! SO ERROR ({tag}): {e}')

    # POS
    row['pos'] = None
    if (row.get('best_pne_welfare') is not None
            and row.get('so_welfare') is not None
            and row['best_pne_welfare'] > 0):
        row['pos'] = compute_pos(row['so_welfare'], row['best_pne_welfare'])

    results_exp1.append(row)

    # Progress
    pne_str = 'BRD-PNE' if brd_pne else 'no-PNE'
    gzr_time_str = f'{row["gzr_time"]:.1f}s' if row.get('gzr_time') is not None else '?s'
    gzr_cuts_str = f'{row["gzr_cuts"]}cuts' if row.get('gzr_cuts') is not None else '?cuts'
    pos_str = f'POS={row["pos"]:.3f}' if row['pos'] is not None else 'POS=N/A'
    fpne_str = f'1stPNE={row["gzr_first_pne_time"]:.1f}s' if row.get('gzr_first_pne_time') is not None else '1stPNE=N/A'
    print(f'[{idx+1}/{len(instances)}] {tag}: {pne_str} | GZR {row.get("gzr_status","?")} '
          f'({gzr_time_str}, {gzr_cuts_str}) | {pos_str} | {fpne_str}')

df_exp1 = pd.DataFrame(results_exp1)
df_exp1.to_csv(exp1_path, index=False)
print(f'\nSaved {len(df_exp1)} rows to {exp1_path.name}')


## 3. Data Loader (for reloading from CSV)

Run cells below to load results and view summaries without re-running the experiment.

In [ ]:
exp1_path = RESULTS_DIR / 'gkg_scalability_exp1.csv'

if exp1_path.exists():
    df_exp1 = pd.read_csv(exp1_path)
    print(f'Loaded {len(df_exp1)} rows from {exp1_path.name}')
    print(f'n values: {sorted(df_exp1["n"].unique())}')
else:
    print(f'File not found: {exp1_path}')

## 4. Summary

In [ ]:
print('=== Scalability Experiment 1 Summary ===')

n_total = len(df_exp1)
n_brd_pne = df_exp1['brd_found_pne'].sum()
n_gzr_opt = (df_exp1['gzr_status'] == 'OPTIMAL').sum()
n_gzr_inf = (df_exp1['gzr_status'] == 'INFEASIBLE').sum()
n_gzr_tl = (df_exp1['gzr_status'] == 'TIME_LIMIT').sum()
n_gzr_err = (df_exp1['gzr_status'] == 'ERROR').sum()
n_pos = df_exp1['pos'].notna().sum()

print(f'Total instances: {n_total}')
print(f'BRD found PNE: {n_brd_pne} ({n_brd_pne/n_total:.1%})')
print(f'GZR OPTIMAL:   {n_gzr_opt} ({n_gzr_opt/n_total:.1%})')
print(f'GZR INFEASIBLE:{n_gzr_inf} ({n_gzr_inf/n_total:.1%})')
print(f'GZR TIME_LIMIT:{n_gzr_tl} ({n_gzr_tl/n_total:.1%})')
print(f'GZR ERROR:     {n_gzr_err} ({n_gzr_err/n_total:.1%})')
print(f'POS computed:  {n_pos} ({n_pos/n_total:.1%})')

In [ ]:
# --- By (n, m) --- main scalability table
print('--- By (n, m) ---')
summary = df_exp1.groupby(['n', 'm']).agg(
    count=('tag', 'count'),
    brd_pne_rate=('brd_found_pne', 'mean'),
    gzr_opt_rate=('gzr_status', lambda x: (x == 'OPTIMAL').mean()),
    gzr_tl_rate=('gzr_status', lambda x: (x == 'TIME_LIMIT').mean()),
    avg_brd_time=('brd_time', 'mean'),
    avg_gzr_time=('gzr_time', 'mean'),
    avg_gzr_cuts=('gzr_cuts', 'mean'),
    avg_gzr_br_calls=('gzr_br_calls', 'mean'),
    avg_first_pne=('gzr_first_pne_time', 'mean'),
    avg_pos=('pos', 'mean'),
).round(4)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 200)
print(summary)


In [ ]:
# --- By (cap_factor, corr) ---
print('--- By (cap_factor, corr) ---')
summary_cf = df_exp1.groupby(['cap_factor', 'corr']).agg(
    count=('tag', 'count'),
    gzr_opt_rate=('gzr_status', lambda x: (x == 'OPTIMAL').mean()),
    gzr_tl_rate=('gzr_status', lambda x: (x == 'TIME_LIMIT').mean()),
    avg_gzr_time=('gzr_time', 'mean'),
    avg_pos=('pos', 'mean'),
).round(4)
print(summary_cf)

In [ ]:
# --- Full breakdown: (n, m, cap_factor, corr) ---
print('--- Full breakdown: (n, m, cap_factor, corr) ---')
full_summary = df_exp1.groupby(['n', 'm', 'cap_factor', 'corr']).agg(
    count=('tag', 'count'),
    brd_pne=('brd_found_pne', 'sum'),
    gzr_status=('gzr_status', 'first'),
    gzr_time=('gzr_time', 'first'),
    gzr_cuts=('gzr_cuts', 'first'),
    gzr_br_calls=('gzr_br_calls', 'first'),
    first_pne=('gzr_first_pne_time', 'first'),
    pos=('pos', 'first'),
).round(4)
pd.set_option('display.max_rows', 200)
print(full_summary)
